# Modelo Basado en Procesos (soluciones steady-state)
## La propuesta: Biogeochemical Flux Model 0D

**Objetivo** Encontrar soluciones steady-state.

**Ventajas** Un modelo basado en procesos respeta las leyes biológicas y físicas; la calibración corrige parámetros y la asimilación el estado con datos reales.

**Desventajas** Requiere calibrar muchos parámetros biológicos difíciles de medir.

In [ ]:
import os,sys
import numpy as np
import scipy.integrate
import pyfabm
import netCDF4 as nc
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks
import math
import subprocess
from scipy.stats import variation
import matplotlib.pyplot as plt
from copy import copy

In [ ]:
# Time derivative
def dy(t0, y):
    model.state[:] = y
    return model.getRates()

In [ ]:
# Ranges to test
ntrials = 6
tre=np.linspace(5, 35, ntrials)
#model.state_variables['N3/n'].value
din=np.linspace(0.2, 0.5, ntrials)
#model.state_variables['N1/p'].value
dip=np.linspace(0.02, 0.05, ntrials)
par=np.linspace(1, 200, ntrials)

In [ ]:
# Time interval: time-integrate over several years (note: FABM's internal time unit is seconds!)
years=2
t_eval = np.linspace(0, 365.*years*86400, 50000*years)
deltaT=t_eval[1]-t_eval[0]
Nt=t_eval.shape[0]
t_last=t_eval[79044:79244,] # last timesteps to save output
Nt_last=t_last.shape[0]

In [ ]:
# Where to save results
fileoutput = 'result_loop4.nc'
f = nc.Dataset(fileoutput, mode='w')

lat_dim = f.createDimension('lat', 1)
lon_dim = f.createDimension('lon', 1)
dep_dim = f.createDimension('z', 1)
time_dim = f.createDimension('time', Nt_last)
trials_dim = f.createDimension('trials', ntrials)
no = f.createDimension('no', 4)

lat = f.createVariable('lat', np.float32, ('lat',))
lat.units = 'degrees_north'
lat.long_name = 'latitude'
f.variables['lat'][:]=0

lon = f.createVariable('lon', np.float32, ('lon',))
lon.units = 'degrees_east'
lon.long_name = 'longitude'
f.variables['lon'][:]=0

time = f.createVariable('time', np.float32, ('time',))
time.units = 'days'
time.long_name = 'time'
f.variables['time'][:]=t_last

depth = f.createVariable('z', np.float32, ('z',))
depth.units = 'meters'
depth.long_name = 'depth'
f.variables['z'][:]=1

trials = f.createVariable('trials', np.float32, ('trials','no',))
trials.units = '-'
trials.long_name = 'dependency'
f.variables['trials'][:,0]=tre
f.variables['trials'][:,1]=din
f.variables['trials'][:,2]=dip
f.variables['trials'][:,3]=par

ndiags=1111
diag_dim = f.createDimension('diag', ndiags)
diag_value = f.createVariable('diag_value', np.float32, ('diag','z','lat','lon','trials','trials','trials'))
#diag_value.units = '-'
diag_value.long_name = 'value diagnostics steady state'

diag_name = f.createVariable('diag_name', 'S20', ('diag',))
#diag_name.units = '-'
diag_name.long_name = 'name diagnostics steady state'

In [ ]:
# Create model (loads fabm.yaml), initializes model.state
model = pyfabm.Model('fabm.yaml')

In [ ]:
# Configure the environment
model.dependencies['cell_thickness'].value = 1.
model.dependencies['practical_salinity'].value = 30.
model.dependencies['temperature'].value = 15.
model.dependencies['density'].value = 1000.
model.dependencies['depth'].value = 1.
model.dependencies['pressure'].value = 1.
#model.dependencies['isBen'].value = 1.
model.dependencies['longitude'].value = 0.
model.dependencies['latitude'].value = 0.
model.dependencies['surface_air_pressure'].value = 1.
model.dependencies['wind_speed'].value = 5.
model.dependencies['mole_fraction_of_carbon_dioxide_in_air'].value = 390.
model.dependencies['number_of_days_since_start_of_the_year'].value = 1.
model.dependencies['surface_downwelling_shortwave_flux'].value = 50.

In [ ]:
# Spectral light dependencies
model.dependencies['surf_direct_downward_irradiance_0250_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0325_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0350_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0375_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0400_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0425_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0450_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0475_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0500_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0525_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0550_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0575_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0600_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0625_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0650_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0675_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0700_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0725_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0775_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0850_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0950_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1050_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1150_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1250_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1350_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1450_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1550_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1650_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1750_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1900_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_2200_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_2900_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_3700_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0250_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0325_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0350_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0375_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0400_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0425_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0450_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0475_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0500_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0525_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0550_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0575_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0600_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0625_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0650_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0675_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0700_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0725_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0775_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0850_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0950_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1050_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1150_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1250_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1350_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1450_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1550_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1650_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1750_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1900_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_2200_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_2900_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_3700_nm'].value = 5.

In [ ]:
model.cell_thickness=1.
# Save model.state for later
#estado = model.state[:] # copy with reference :(
estado = copy(model.state[:]) # copy without reference

In [ ]:
# Loop temperature
for s in range(ntrials):
    val1 = tre[s]
    model.dependencies['temperature'].value = val1
    # Verify the model is ready to be used
    #assert model.checkReady()

    # Loop DIN
    for w in range(ntrials):
        val2 = din[w]
        model.state_variables['N3/n'].value = val2
        # Verify the model is ready to be used
        #assert model.checkReady()

        # Loop DIP
        for x in range(ntrials):
            val3 = dip[x]
            model.state_variables['N1/p'].value = val3
            # Verify the model is ready to be used
            assert model.checkReady()

            # Time-integrate over n years (note: FABM's internal time unit is seconds!)
            # INITIAL VALUE SOLVER
            sol = scipy.integrate.solve_ivp(dy, [0., 365.*years*86400], model.state, t_eval=t_eval)
            #sol.t.shape
            t = sol.t/86400
            y = sol.y.T
            cual=sol.t.shape[0]-1

            # ODE INT
            #sol = scipy.integrate.odeint(dy, model.state, t_eval) # returns only y
            ##sol = scipy.integrate.OdeSolver(dy, 0., model.state, 365.*years*86400, False)
            ##sol.shape
            #t = t_eval/86400
            #y = sol
            #cual=t.shape[0]-1

            # Update model.state
            model.state[:]=y[cual,:]
            print({cual})

            # Save results state variables
            for v,variable in enumerate(model.state_variables):
                ncvar = variable.name.replace("/","_")
                if  s == 0 and w == 0 and x == 0 :
                    #print('enters here')
                    var = f.createVariable(ncvar, np.float32, ('time','z','lat','lon','trials','trials','trials'))
                    var.units = variable.units
                    var.long_name = variable.long_name
                    #var = None
                if cual >= 200 :
                    f.variables[ncvar][:,0,0,0,s,w,x]=y[(cual-200):cual,v] # only last 200 timesteps
                else :
                    f.variables[ncvar][:,0,0,0,s,w,x]=np.zeros((200,1,1,1,1,1))

            # Getting diagnostic variables
            list1=['test']
            n=0
            list2=[1]
            substrings = ['_et','_iN','_tN','_eiPPY','_sum','_rugc','_run','_rrc','_rum']
            for v,variable in enumerate(model.diagnostic_variables):
                ncvar = variable.name.replace("/","_")
                if any(substring in ncvar for substring in substrings):
                    if model.diagnostic_variables[variable.name].value is None:
                        #print(" output is None! ")
                        dummy=0
                    else:
                        #print(f"  {variable.name} = {variable.long_name} ({variable.units})")
                        list1.append(variable.name)
                        n=n+1
                        if  s == 0 and w == 0 and x == 0 :
                            f.variables['diag_name'][n]=variable.name
                        ejemplo=model.diagnostic_variables[variable.name].value
                        list2.append(ejemplo.item())
            # names,
            if  s == 0 and w == 0 and x == 0 :
                diag_value.units = list1

             # and save values
            list2 = [-999 if math.isnan(b) else b for b in list2]
            f.variables['diag_value'][:,0,0,0,s,w,x]=list2
            print(f"TRE = {tre[s]}")
            print(f"DIN = {din[w]}")
            print(f"DIP = {dip[x]}")
            #print(f"PAR = {par[z]}")
            model.state[:]=estado
            assert model.checkReady()
            print(s)
            print(w)
            print(x)
        model.state[:]=estado
        assert model.checkReady()
    model.state[:]=estado
    assert model.checkReady()                      